# **Notebook 03 — Data Cleaning**

## Objectives

* Investigate data quality: missing values, duplicates, outliers
* Document all findings and handling decisions per CRISP-DM Data Preparation
* Perform stratified train/test split preserving class ratios
* Save clean splits to versioned output folder

## Inputs

* `data/creditcard.csv`

## Outputs

* `outputs/v1/X_train.csv`, `outputs/v1/X_test.csv`
* `outputs/v1/y_train.csv`, `outputs/v1/y_test.csv`
* Documentation of all data quality decisions

---
## Change working directory

In [1]:
import os

current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    os.chdir(os.path.dirname(current_dir))
print(f"Working directory: {os.getcwd()}")

Working directory: C:\Users\name\Desktop\All IN\Code Inst. Projects\credit-card-fraud-detection


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/creditcard.csv")
print(f"Loaded dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")

Loaded dataset: 284,807 rows × 31 columns


---
## 1. Missing Values Investigation

Checking every column for missing values. Even if we expect none (PCA-processed data), documenting this investigation is part of the CRISP-DM Data Preparation step.

In [3]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

print("Missing Values Analysis")
print("=" * 50)

if missing.sum() == 0:
    print("No missing values found across all 31 features.")
    print("No imputation required.")
else:
    print(f"Total missing: {missing.sum()}")
    print(missing[missing > 0])

Missing Values Analysis
No missing values found across all 31 features.
No imputation required.


### Missing Values Conclusion

The dataset contains zero missing values across all 31 features. This is expected as the data has been pre-processed with PCA by the original data providers (ULB research group). No imputation is required.

While no action is needed, documenting this investigation demonstrates due diligence as part of the CRISP-DM Data Preparation step.

---
## 2. Duplicate Investigation

Checking for exact duplicate rows. In transaction data, exact duplicates are suspicious — it is unlikely that two completely different transactions would have identical values across all 31 features including Time and Amount.

In [4]:
duplicates = df.duplicated().sum()

print("Duplicate Analysis")
print("=" * 50)
print(f"Duplicate rows found: {duplicates}")

if duplicates > 0:
    # Investigate which class the duplicates belong to
    dup_mask = df.duplicated(keep=False)
    dup_fraud = df[dup_mask & (df['Class'] == 1)].shape[0]
    dup_legit = df[dup_mask & (df['Class'] == 0)].shape[0]

    print(f"  Duplicates in fraud class:  {dup_fraud}")
    print(f"  Duplicates in legit class:  {dup_legit}")
    print()

    # Show a few examples
    print("Example duplicate rows:")
    dup_examples = df[df.duplicated(keep='last')].head(3)
    print(dup_examples[['Time', 'Amount', 'Class']].to_string())

Duplicate Analysis
Duplicate rows found: 1081
  Duplicates in fraud class:  32
  Duplicates in legit class:  1822

Example duplicate rows:
     Time  Amount  Class
32   26.0    6.14      0
34   26.0    1.77      0
112  74.0    1.18      0


In [5]:
# Remove duplicates
if duplicates > 0:
    shape_before = df.shape[0]
    df = df.drop_duplicates()
    shape_after = df.shape[0]
    removed = shape_before - shape_after

    print(f"Removed {removed} duplicate rows")
    print(f"Dataset: {shape_before:,} → {shape_after:,} rows")
else:
    print("No duplicates to remove.")

Removed 1081 duplicate rows
Dataset: 284,807 → 283,726 rows


### Duplicate Handling Conclusion

Duplicate rows were investigated. Since PCA features combined with Time and Amount make exact duplicates extremely unlikely to represent genuinely different transactions, duplicates were removed to prevent potential data leakage between the train and test sets.

---
## 3. Outlier Analysis

Only `Amount` and `Time` are in their original scale — V1-V28 are PCA-transformed and their scale is expected. We investigate outliers but must be careful about removing them in a fraud detection context.

In [6]:
print("Outlier Analysis (IQR Method)")
print("=" * 50)

for col in ['Amount', 'Time']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_pct = outliers / len(df) * 100

    print(f"\n{col}:")
    print(f"  Q1 = {Q1:.2f}, Q3 = {Q3:.2f}, IQR = {IQR:.2f}")
    print(f"  Lower bound = {lower:.2f}, Upper bound = {upper:.2f}")
    print(f"  Outliers: {outliers:,} ({outlier_pct:.2f}%)")

Outlier Analysis (IQR Method)

Amount:
  Q1 = 5.60, Q3 = 77.51, IQR = 71.91
  Lower bound = -102.27, Upper bound = 185.38
  Outliers: 31,685 (11.17%)

Time:
  Q1 = 54204.75, Q3 = 139298.00, IQR = 85093.25
  Lower bound = -73435.12, Upper bound = 266937.88
  Outliers: 0 (0.00%)


In [7]:
# Check how many outlier amounts are in fraud vs legit
Q1 = df['Amount'].quantile(0.25)
Q3 = df['Amount'].quantile(0.75)
IQR = Q3 - Q1
upper = Q3 + 1.5 * IQR

outlier_mask = df['Amount'] > upper

print("Amount outliers by class:")
print(f"  Legitimate: {df[outlier_mask & (df['Class'] == 0)].shape[0]}")
print(f"  Fraud:      {df[outlier_mask & (df['Class'] == 1)].shape[0]}")

Amount outliers by class:
  Legitimate: 31598
  Fraud:      87


### Outlier Decision

Amount shows outliers by the IQR method. However, these are **NOT removed** because:

1. **High-value legitimate transactions** are normal business behaviour (large purchases, payments)
2. **Fraud amounts** may naturally fall outside typical ranges — removing them could eliminate important patterns
3. **RobustScaler** (used in Feature Engineering) is resistant to outliers by design
4. In fraud detection, unusual values are often the **signal** rather than noise

This decision preserves the full distribution for the ML model to learn from.